# Silver Layer — HCPCS Reference Dictionary (`d_hcpcs`)

**Source -> Target:** `<source_schema>.d_hcpcs` -> `<target_schema>.d_hcpcs` (defaults `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`). The target is replaced via `CREATE OR REPLACE`; downstream Gold depends on it.

**Objective:** Standardize HCPCS reference codes, categories and descriptions while preserving their original clinical meaning.

**Expected grain:** One record per HCPCS code (`code`). `code` is documented as CHAR(5) NOT NULL and links to `hcpcsevents.hcpcs_cd` in MIMIC-IV `hosp.d_hcpcs`; uniqueness is asserted before publication.

**Silver rules:** Preserve all Bronze records, standardize data types and whitespace, validate code completeness and uniqueness, and reconcile Bronze–Silver counts. No deletion, deduplication, filtering, imputation or aggregation.

In [0]:
# Import PySpark functions
from pyspark.sql import functions as F

In [0]:
# Configure Bronze source and Silver target schemas (widgets allow standalone and dbutils.notebook.run use)
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()
ENFORCE_PROFILED_COUNTS = dbutils.widgets.get("enforce_profiled_counts").strip().lower() == "true"

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

TABLE = "d_hcpcs"
SOURCE_TABLE = f"{BRONZE}.{TABLE}"
TARGET_TABLE = f"{SILVER}.{TABLE}"

# Expected published contract (ordered column -> type)
EXPECTED_SCHEMA = {
    "code": "string",
    "category": "int",
    "long_description": "string",
    "short_description": "string",
}

In [0]:
# Load and inspect the HCPCS reference dictionary
bronze_df = spark.table(SOURCE_TABLE)

bronze_df.printSchema()
display(bronze_df.limit(20))

print("Bronze rows:", bronze_df.count())

root
 |-- code: string (nullable = true)
 |-- category: integer (nullable = true)
 |-- long_description: string (nullable = true)
 |-- short_description: string (nullable = true)



code,category,long_description,short_description
00000,null,null,Invalid Code
0001F,2,null,Composite measures
0002F,2,null,Composite measures
0003F,2,null,Composite measures
0004F,2,null,Composite measures
0005F,2,null,Composite measures
0006F,2,null,Composite measures
0007F,2,null,Composite measures
0008F,2,null,Composite measures
0009F,2,null,Composite measures


Bronze rows: 89208


In [0]:
# Profile source row counts and missing values
bronze_profile = bronze_df.select(
    F.count("*").alias("total_rows"),
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in bronze_df.columns
    ]
)

display(bronze_profile)

total_rows,null_code,null_category,null_long_description,null_short_description
89208,0,6850,82408,0


In [0]:
# Standardize HCPCS codes, categories and descriptions
silver_df = bronze_df.select(
    F.trim(F.col("code").cast("string")).alias("code"),
    F.col("category").cast("int").alias("category"),
    F.trim(F.col("long_description").cast("string")).alias("long_description"),
    F.trim(F.col("short_description").cast("string")).alias("short_description")
)

silver_df.printSchema()
display(silver_df.limit(20))

root
 |-- code: string (nullable = true)
 |-- category: integer (nullable = true)
 |-- long_description: string (nullable = true)
 |-- short_description: string (nullable = true)



code,category,long_description,short_description
00000,null,null,Invalid Code
0001F,2,null,Composite measures
0002F,2,null,Composite measures
0003F,2,null,Composite measures
0004F,2,null,Composite measures
0005F,2,null,Composite measures
0006F,2,null,Composite measures
0007F,2,null,Composite measures
0008F,2,null,Composite measures
0009F,2,null,Composite measures


In [0]:
# Check repeated HCPCS codes without removing records
duplicate_df = (
    silver_df
    .groupBy("code")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicate_df.count()

print("Duplicate HCPCS code groups:", duplicate_count)
display(duplicate_df.limit(20))

Duplicate HCPCS code groups: 0


code,count


In [0]:
# Check missing codes, categories and descriptions
quality_df = silver_df.select(
    *[
        F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
        for c in silver_df.columns
    ],
    F.sum(
        F.when(F.col("code") == "", 1).otherwise(0)
    ).alias("blank_codes"),
    F.sum(
        F.when(F.col("long_description") == "", 1).otherwise(0)
    ).alias("blank_long_descriptions"),
    F.sum(
        F.when(F.col("short_description") == "", 1).otherwise(0)
    ).alias("blank_short_descriptions")
)

display(quality_df)

null_code,null_category,null_long_description,null_short_description,blank_codes,blank_long_descriptions,blank_short_descriptions
0,6850,82408,0,0,0,0


In [0]:
# Inspect HCPCS categories and record distribution
category_profile = (
    silver_df
    .groupBy("category")
    .agg(
        F.count("*").alias("records"),
        F.countDistinct("code").alias("distinct_codes")
    )
    .orderBy("category")
)

display(category_profile)

category,records,distinct_codes
null,6850,6850
1,80452,80452
2,1391,1391
3,515,515


In [0]:
# Verify that all Bronze rows are preserved
bronze_count = bronze_df.count()
silver_count = silver_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)

assert bronze_count == silver_count, "Bronze–Silver row-count mismatch"

Bronze rows: 89208
Silver rows: 89208


In [0]:
# Validate HCPCS keys and record preservation
quality = quality_df.first().asDict()

assert duplicate_count == 0, f"Duplicate HCPCS codes: {duplicate_count}"
assert quality["null_code"] == 0, "Missing HCPCS codes"
assert quality["blank_codes"] == 0, "Blank HCPCS codes"
assert bronze_count == silver_count, "Record counts do not match"

print("All critical Silver quality checks passed.")

All critical Silver quality checks passed.


In [0]:
# Publish the validated Silver HCPCS dictionary (CREATE OR REPLACE; downstream Gold and hcpcsevents coverage checks depend on it)
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

silver_df.writeTo(TARGET_TABLE).using("delta").createOrReplace()

print("Published:", TARGET_TABLE)

Published: cdr_mimiciv.silver.d_hcpcs


In [0]:
# Verify the published HCPCS dictionary
published_df = spark.table(TARGET_TABLE)

published_count = published_df.count()

print("Bronze rows:", bronze_count)
print("Silver rows:", silver_count)
print("Published rows:", published_count)

assert published_count == silver_count, "Published row-count mismatch"
published_schema = {f.name: f.dataType.simpleString() for f in published_df.schema.fields}
assert list(published_schema.items()) == list(EXPECTED_SCHEMA.items()), (
    f"Published schema differs from contract: {published_schema}"
)

published_keys = published_df.select(
    F.sum(F.col("code").isNull().cast("int")).alias("null_codes"),
    F.countDistinct("code").alias("distinct_codes")
).first()
assert published_keys["null_codes"] == 0, "Published HCPCS codes contain NULLs"
assert published_keys["distinct_codes"] == published_count, "Published HCPCS codes are not unique"

display(published_df.limit(20))

print("Silver d_hcpcs table verified successfully.")

Bronze rows: 89208
Silver rows: 89208
Published rows: 89208


code,category,long_description,short_description
00000,null,null,Invalid Code
0001F,2,null,Composite measures
0002F,2,null,Composite measures
0003F,2,null,Composite measures
0004F,2,null,Composite measures
0005F,2,null,Composite measures
0006F,2,null,Composite measures
0007F,2,null,Composite measures
0008F,2,null,Composite measures
0009F,2,null,Composite measures


Silver d_hcpcs table verified successfully.


In [0]:
# Finish successfully only after every assertion above has passed
dbutils.notebook.exit(f"{TABLE}:OK")